In [18]:
# 1.1 Import Library
import pandas as pd
import numpy as np
import random
import os

print("Library berhasil dimuat")

Library berhasil dimuat


In [19]:
# 1.2 Parameter Sampling (Stratified Sampling)
TOTAL_SAMPLES_PER_CATEGORY = 150
CATEGORIES = {
    'pendek': {'min': 0, 'max': 49, 'label': '< 50 karakter'},
    'sedang': {'min': 50, 'max': 149, 'label': '50-149 karakter'},
    'panjang': {'min': 150, 'max': float('inf'), 'label': '>= 150 karakter'}
}
TOTAL_SAMPLES = TOTAL_SAMPLES_PER_CATEGORY * len(CATEGORIES)

print("PARAMETER SAMPLING (STRATIFIED SAMPLING)")
print(f"Total sampel yang akan diambil: {TOTAL_SAMPLES} tweets")
print(f"Sampel per kategori: {TOTAL_SAMPLES_PER_CATEGORY} tweets")
print("\nKategori panjang teks:")
for cat_name, cat_info in CATEGORIES.items():
    print(f"  - {cat_name.capitalize()}: {cat_info['label']}")

PARAMETER SAMPLING (STRATIFIED SAMPLING)
Total sampel yang akan diambil: 450 tweets
Sampel per kategori: 150 tweets

Kategori panjang teks:
  - Pendek: < 50 karakter
  - Sedang: 50-149 karakter
  - Panjang: >= 150 karakter


In [20]:
# 1.3 Load Data Preprocessing Final
# Load dari data preprocessing
df = pd.read_csv('../../datapreprocessingcopy/data_preprocessing_final.csv')

print(f"\nData preprocessing berhasil dimuat: {len(df)} tweets")
print(f"Kolom: {df.columns.tolist()}")
df.head()


Data preprocessing berhasil dimuat: 13192 tweets
Kolom: ['no', 'timestamp', 'teks', 'teks_processed']


,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,tertib media online DPR pemerintah jangan spor...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus evaluasi lagi kebijakan bebas visa utama...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang pengaturan logis apa undang un...
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,bebas suara dapat memang jamin UU tetapi bebas...


In [21]:
# 1.4 Pengelompokan Data Berdasarkan Panjang Teks
print("\nPENGELOMPOKAN DATA BERDASARKAN PANJANG TEKS")

df = df.copy()
df['text_length'] = df['teks_processed'].apply(len)

def categorize_text_length(length):
    if length < 50:
        return 'pendek'
    elif length < 150:
        return 'sedang'
    else:
        return 'panjang'

df['length_category'] = df['text_length'].apply(categorize_text_length)

grouping_stats = df['length_category'].value_counts().to_dict()

print("\nHasil pengelompokan seluruh data:")
for cat_name in ['pendek', 'sedang', 'panjang']:
    count = grouping_stats.get(cat_name, 0)
    percentage = (count / len(df)) * 100
    print(f"  Kategori '{cat_name}': {count} tweets ({percentage:.1f}%)")

# Simpan data pengelompokan ke CSV
df_grouping = df[['no', 'timestamp', 'teks', 'teks_processed', 'text_length', 'length_category']].copy()

os.makedirs('../outputs/evaluation', exist_ok=True)

df_grouping.to_csv('../outputs/evaluation/data_grouping_by_length.csv', 
                   index=False, encoding='utf-8')

print(f"\nFile pengelompokan disimpan: ../outputs/evaluation/data_grouping_by_length.csv")


PENGELOMPOKAN DATA BERDASARKAN PANJANG TEKS

Hasil pengelompokan seluruh data:
  Kategori 'pendek': 1214 tweets (9.2%)
  Kategori 'sedang': 9254 tweets (70.1%)
  Kategori 'panjang': 2724 tweets (20.6%)

File pengelompokan disimpan: ../outputs/evaluation/data_grouping_by_length.csv


In [22]:
# 1.5 Stratified Sampling Berdasarkan Kelompok
random.seed(42)

print("\nSAMPLING STRATIFIED BERDASARKAN KELOMPOK PANJANG TEKS")

samples_list = []
sampling_stats = {}

print("\nProses sampling per kategori:")

for cat_name, cat_info in CATEGORIES.items():
    df_category = df[df['length_category'] == cat_name].copy()
    
    total_in_category = len(df_category)
    
    if total_in_category < TOTAL_SAMPLES_PER_CATEGORY:
        print(f"  WARNING: Kategori '{cat_name}' hanya memiliki {total_in_category} tweets")
        print(f"  Mengambil semua data yang tersedia...")
        df_sample_category = df_category.sample(frac=1, random_state=42)
        actual_samples = total_in_category
    else:
        df_sample_category = df_category.sample(n=TOTAL_SAMPLES_PER_CATEGORY, random_state=42)
        actual_samples = TOTAL_SAMPLES_PER_CATEGORY
    
    samples_list.append(df_sample_category)
    
    sampling_stats[cat_name] = {
        'total_available': total_in_category,
        'samples_taken': actual_samples,
        'percentage': f"{(total_in_category/len(df)*100):.1f}%"
    }
    
    print(f"  Kategori '{cat_name}': {actual_samples} tweets diambil "
          f"(dari {total_in_category} total, {sampling_stats[cat_name]['percentage']} populasi)")

df_sample_final = pd.concat(samples_list).reset_index(drop=True)

print(f"\nTotal sampel akhir: {len(df_sample_final)} tweets")
print(f"Jumlah unik (no): {df_sample_final['no'].nunique()} tweets")


SAMPLING STRATIFIED BERDASARKAN KELOMPOK PANJANG TEKS

Proses sampling per kategori:
  Kategori 'pendek': 150 tweets diambil (dari 1214 total, 9.2% populasi)
  Kategori 'sedang': 150 tweets diambil (dari 9254 total, 70.1% populasi)
  Kategori 'panjang': 150 tweets diambil (dari 2724 total, 20.6% populasi)

Total sampel akhir: 450 tweets
Jumlah unik (no): 450 tweets


In [23]:
# 1.6 Export Template untuk Labeling Manual
export_cols = ['no', 'timestamp', 'teks', 'teks_processed', 'text_length', 'length_category']

export_df = df_sample_final[export_cols].copy()

export_df['ground_truth_score'] = ''
export_df['ground_truth_category'] = ''
export_df['notes'] = ''

export_df.to_csv('../outputs/evaluation/ground_truth.csv',
                 index=False, encoding='utf-8')

print("\nTEMPLATE BERHASIL DIBUAT")
print(f"Lokasi file: ../outputs/evaluation/ground_truth.csv")
print(f"Total baris: {len(export_df)} tweets")


TEMPLATE BERHASIL DIBUAT
Lokasi file: ../outputs/evaluation/ground_truth.csv
Total baris: 450 tweets


In [24]:
# 1.7 Panduan Labeling Manual
# DISUNTAI SESUAI DENGAN GAMBAR VADER YANG DIBERIKAN
print("\nPANDUAN LABELING MANUAL - SKALA SENTIMEN")

print("""
PENJELASAN:
Anda akan memberikan label berdasarkan skala intensitas sentimen (VADER Scale).
Berikan nilai integer dari -4 sampai +4.

SKALA LABELING:

NEGATIF (Negative):
[-4] Extremely Negative  (Sangat Ekstrem Negatif)
[-3] Very Negative       (Sangat Negatif)
[-2] Moderately Negative (Negatif Sedang)
[-1] Slightly Negative   (Negatif Sedikit)

NETRAL (Neutral):
[ 0] Neutral (or Neither, N/A)

POSITIF (Positive):
[ 1] Slightly Positive   (Positif Sedikit)
[ 2] Moderately Positive (Positif Sedang)
[ 3] Very Positive       (Sangat Positif)
[ 4] Extremely Positive  (Sangat Ekstrem Positif)

KRITERIA PENILAIAN:

A. EXTREMELY NEGATIVE (-4)
   - Kemarahan meledak, hinaan kasar, penolakan mutlak.
   - Contoh: "DPR sampah!", "Bubarkan saja!"

B. VERY NEGATIVE (-3)
   - Kekecewaan mendalam, kritik tajam, sindiran keras.
   - Contoh: "Kinerja sangat mengecewakan", "Kebijakan ini merusak."

C. MODERATELY NEGATIVE (-2)
   - Ketidakpuasan jelas, keluhan spesifik, pesimisme.
   - Contoh: "Sayang sekali hasilnya buruk", "Tidak ada progres berarti."

D. SLIGHTLY NEGATIVE (-1)
   - Keraguan, kritik ringan, harapan yang sedikit pudar.
   - Contoh: "Seharusnya bisa lebih baik", "Kurang optimal."

E. NEUTRAL (0)
   - Fakta objektif, berita tanpa opini, pertanyaan netral.
   - Contoh: "Rapat dimulai pukul 10.00 WIB."

F. SLIGHTLY POSITIVE (+1)
   - Harapan, apresiasi ringan, dukungan sopan.
   - Contoh: "Semoga ke depan lebih baik", "Terima kasih."

G. MODERATELY POSITIVE (+2)
   - Pujian jelas, optimisme, dukungan aktif.
   - Contoh: "Kerja bagus DPR!", "Kebijakan ini membantu rakyat."

H. VERY POSITIVE (+3)
   - Antusiasme tinggi, pujian kuat, kebanggaan.
   - Contoh: "Luar biasa! Ini yang kami tunggu."

I. EXTREMELY POSITIVE (+4)
   - Euforia total, dukungan fanatik, kata-kata sangat positif.
   - Contoh: "DPR terbaik sepanjang masa!", "Sempurna!"

CATATAN PENTING:
- Labelilah berdasarkan KONTEN dan INTENSI tweet.
- Jika ragu, beri catatan di kolom 'notes'.
""")


PANDUAN LABELING MANUAL - SKALA SENTIMEN

PENJELASAN:
Anda akan memberikan label berdasarkan skala intensitas sentimen (VADER Scale).
Berikan nilai integer dari -4 sampai +4.

SKALA LABELING:

NEGATIF (Negative):
[-4] Extremely Negative  (Sangat Ekstrem Negatif)
[-3] Very Negative       (Sangat Negatif)
[-2] Moderately Negative (Negatif Sedang)
[-1] Slightly Negative   (Negatif Sedikit)

NETRAL (Neutral):
[ 0] Neutral (or Neither, N/A)

POSITIF (Positive):
[ 1] Slightly Positive   (Positif Sedikit)
[ 2] Moderately Positive (Positif Sedang)
[ 3] Very Positive       (Sangat Positif)
[ 4] Extremely Positive  (Sangat Ekstrem Positif)

KRITERIA PENILAIAN:

A. EXTREMELY NEGATIVE (-4)
   - Kemarahan meledak, hinaan kasar, penolakan mutlak.
   - Contoh: "DPR sampah!", "Bubarkan saja!"

B. VERY NEGATIVE (-3)
   - Kekecewaan mendalam, kritik tajam, sindiran keras.
   - Contoh: "Kinerja sangat mengecewakan", "Kebijakan ini merusak."

C. MODERATELY NEGATIVE (-2)
   - Ketidakpuasan jelas, keluhan 

In [25]:
# 1.8 Preview Sample untuk Verifikasi
print("\nPREVIEW 5 SAMPLE TWEET UNTUK VERIFIKASI")

for i in range(5):
    row = export_df.iloc[i]
    print(f"\nTweet {i+1} (No: {row['no']})")
    print(f"Panjang: {row['text_length']} karakter")
    print(f"Kategori: {row['length_category']}")
    print(f"Teks: {row['teks'][:100]}...")
    print("-" * 70)


PREVIEW 5 SAMPLE TWEET UNTUK VERIFIKASI

Tweet 1 (No: 6885)
Panjang: 48 karakter
Kategori: pendek
Teks: meminta untuk fokus dalam membahas penyelesaian RUU Pemilu...
----------------------------------------------------------------------

Tweet 2 (No: 7320)
Panjang: 49 karakter
Kategori: pendek
Teks: Rapat Paripurna DPR RI Setujui Dua Anggota BPK Terpilih...
----------------------------------------------------------------------

Tweet 3 (No: 360)
Panjang: 35 karakter
Kategori: pendek
Teks: RUU Ketentuan Umum Perpajakan Masih Dibahas DPR...
----------------------------------------------------------------------

Tweet 4 (No: 1514)
Panjang: 49 karakter
Kategori: pendek
Teks: Berlanjut Baleg DPR RI Bahas DIM RUU Kekarantinaan Kesehatan...
----------------------------------------------------------------------

Tweet 5 (No: 4881)
Panjang: 35 karakter
Kategori: pendek
Teks: Rapat Paripurna DPR RI Setujui UU APBN...
----------------------------------------------------------------------


In [26]:
# 1.9 Statistik Sampel untuk Dokumentasi
print("\nSTATISTIK SAMPLING GROUND TRUTH")

length_stats = {
    'min': int(export_df['text_length'].min()),
    'max': int(export_df['text_length'].max()),
    'mean': float(export_df['text_length'].mean()),
    'median': float(export_df['text_length'].median()),
    'std': float(export_df['text_length'].std())
}

print(f"\nPanjang Teks (karakter):")
print(f"  Min: {length_stats['min']}")
print(f"  Max: {length_stats['max']}")
print(f"  Mean: {length_stats['mean']:.1f}")
print(f"  Median: {length_stats['median']:.1f}")
print(f"  Std Dev: {length_stats['std']:.1f}")

print(f"\nDistribusi Sampel per Kategori:")
for cat_name, stats in sampling_stats.items():
    print(f"  {cat_name.capitalize()} ({CATEGORIES[cat_name]['label']}):")
    print(f"    - Sampel diambil: {stats['samples_taken']}")
    print(f"    - Total di populasi: {stats['total_available']} ({stats['percentage']})")

import json
with open('../outputs/evaluation/sampling_stats.json', 'w') as f:
    json.dump({
        'total_samples': len(export_df),
        'sampling_method': 'stratified_sampling_by_text_length',
        'samples_per_category': TOTAL_SAMPLES_PER_CATEGORY,
        'category_definitions': {
            cat_name: {
                'range': f"{cat_info['min']}-{cat_info['max'] if cat_info['max'] != float('inf') else 'infinity'}",
                'label': cat_info['label']
            }
            for cat_name, cat_info in CATEGORIES.items()
        },
        'grouping_statistics': {
            cat_name: {
                'total_in_population': grouping_stats.get(cat_name, 0),
                'samples_taken': sampling_stats[cat_name]['samples_taken']
            }
            for cat_name in CATEGORIES.keys()
        },
        'sampling_statistics': sampling_stats,
        'length_statistics': length_stats,
        'note': 'Slovin formula tidak digunakan (tidak baku menurut dosen pembimbing)'
    }, f, indent=2)

print("\nStatistik disimpan: ../outputs/evaluation/sampling_stats.json")

print("\nPROSES SELESAI")
print(f"File yang dihasilkan:")
print(f"  1. ../outputs/evaluation/data_grouping_by_length.csv (seluruh data dengan kategori)")
print(f"  2. ../outputs/evaluation/ground_truth.csv (450 sampel untuk labeling)")
print(f"  3. ../outputs/evaluation/sampling_stats.json (statistik dokumentasi)")


STATISTIK SAMPLING GROUND TRUTH

Panjang Teks (karakter):
  Min: 13
  Max: 267
  Mean: 112.0
  Median: 91.5
  Std Dev: 73.3

Distribusi Sampel per Kategori:
  Pendek (< 50 karakter):
    - Sampel diambil: 150
    - Total di populasi: 1214 (9.2%)
  Sedang (50-149 karakter):
    - Sampel diambil: 150
    - Total di populasi: 9254 (70.1%)
  Panjang (>= 150 karakter):
    - Sampel diambil: 150
    - Total di populasi: 2724 (20.6%)

Statistik disimpan: ../outputs/evaluation/sampling_stats.json

PROSES SELESAI
File yang dihasilkan:
  1. ../outputs/evaluation/data_grouping_by_length.csv (seluruh data dengan kategori)
  2. ../outputs/evaluation/ground_truth.csv (450 sampel untuk labeling)
  3. ../outputs/evaluation/sampling_stats.json (statistik dokumentasi)
